# Penalties Analysis

Common setup and helpers are defined once here.

Data: auto-loads every `{chain}_*.csv` in `PENALTIES_DATA_DIR` and groups by `blockchain`. Cross-chain views activate automatically once >1 chain is present.

In [ ]:
# === COMMON SETUP & HELPERS (defined once) ===
import glob, os, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 80)
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True, 'grid.alpha': 0.3})
DATA_DIR = os.environ.get('PENALTIES_DATA_DIR', '.')
NIL = {'<nil>': np.nan, 'None': np.nan, 'nan': np.nan, '': np.nan}
WEI = 1e18

NUMERIC = ['volume_native','slippage_native','reward_penalty_native','reward_penalty_uncapped_native',
    'reward_native','penalty_native','penalty_uncapped_native','penalty_cap_native','reward_cap_upper_native',
    'reference_score','observed_score','execution_cost_native','slippage_usd','order_size_usd','markout_usd',
    'markout_relative','slippage_tolerance_bps','calculated_slippage_bps','seconds_since_created',
    'seconds_to_settle','auction_id','settlement_block','block_deadline']
BOOL = ['settled','is_excluded_from_penalties','is_quote_reward_eligible','partially_fillable',
    'is_out_of_market','smart_slippage']
# whole-token (/WEI) THEN USD (avoids 1e18 blowup with a per-token price)
ECON_NATIVE = ['reward_native','penalty_native','penalty_uncapped_native','penalty_cap_native',
    'reward_penalty_native','reward_cap_upper_native','execution_cost_native','reference_score','observed_score']

def _num(s): return pd.to_numeric(s.astype(str).replace(NIL), errors='coerce')

def load_chain_csvs(data_dir=DATA_DIR, pattern='*_*_*.csv'):
    """Load + concat every chain CSV; print provenance (name, rows, md5) to pin the extract."""
    paths = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not paths: raise FileNotFoundError(f'no CSVs matching {pattern} in {data_dir!r}')
    frames = []
    for p in paths:
        d = pd.read_csv(p, low_memory=False); d['_source_file'] = os.path.basename(p)
        md5 = hashlib.md5(open(p, 'rb').read()).hexdigest()[:12]
        print(f'  {os.path.basename(p):45s} rows={len(d):>7,}  md5={md5}')
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

def native_token_usd_price(df):
    """Per-chain-day median native-token USD price from settled rows (order_size_usd / volume_tok),
    applied to all rows incl reverts. Stable vs noisy per-row slippage ratios. Falls back to per-chain."""
    s = df['settled'] == True
    vol_tok = df['volume_native'] / WEI
    px = (df['order_size_usd'] / vol_tok).where(s & (vol_tok > 0))
    day = pd.to_datetime(df['auction_timestamp'], errors='coerce').dt.date
    key = pd.DataFrame({'chain': df['blockchain'], 'day': day, 'px': px})
    cd = key.groupby(['chain', 'day'])['px'].transform('median')
    ch = key.groupby('chain')['px'].transform('median')
    return cd.fillna(ch)

def prepare(df):
    """Coerce types, derive native→USD price, build *_tok/*_usd economic cols, two P&L concepts, bps, flags.
    Grains: attempt = (auction_id, order_uid); reward/penalty = (auction_id, solver); markout/size_usd = settled-only."""
    df = df.copy()
    for c in NUMERIC:
        if c in df: df[c] = _num(df[c])
    for c in BOOL:
        if c in df: df[c] = df[c].astype(str).str.lower().map({'true': True, 'false': False})
    df['native_usd_price'] = native_token_usd_price(df)             # USD per whole token
    for c in ECON_NATIVE:                                           # wei -> token -> USD
        base = c[:-len('_native')] if c.endswith('_native') else c
        df[base + '_tok'] = df[c] / WEI
        df[base + '_usd'] = df[base + '_tok'] * df['native_usd_price']
    df['volume_tok'] = df['volume_native'] / WEI
    df['volume_usd'] = df['volume_tok'] * df['native_usd_price']
    # size_usd: Dune order_size_usd is settled-only; fall back to volume_usd (present on reverts)
    df['size_usd'] = df['order_size_usd'].where(df['order_size_usd'].notna(), df['volume_usd'])
    # GROSS excludes reward/penalty. Using when penalty/cap is the regressor (else tautological).
    # NET includes them. Realised welfare under the mechanism.
    df['gross_execution_pnl_usd'] = np.where(df['settled'] == True, df['slippage_usd'] - df['execution_cost_usd'], np.nan)
    df['net_mechanism_pnl_usd'] = (df['reward_penalty_usd'].fillna(0) + df['slippage_usd'].fillna(0) - df['execution_cost_usd'].fillna(0))
    df['reverted'] = ~df['settled'].fillna(False)
    df['cap_binds'] = df['penalty_uncapped_native'] > df['penalty_cap_native']
    df['is_solo_bidder'] = df['reference_score'] == 0              # reference_score == 0 = penalty structurally 0
    df['markout_bps'] = df['markout_relative'] * 1e4
    df['penalty_bps'] = (df['penalty_usd'] / df['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    df['reward_bps'] = (df['reward_usd'] / df['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    return df

def apply_filters(df, fok_only=True, in_market_only=True, exclude_penalty_excluded=True):
    """In-market fill-or-kill only; reverted winners KEPT (the signal). Returns (df, funnel)."""
    funnel = {'start': len(df)}; out = df
    if fok_only: out = out[out['partially_fillable'] == False]; funnel['fill_or_kill'] = len(out)
    if in_market_only: out = out[out['is_out_of_market'] == False]; funnel['in_market'] = len(out)
    if exclude_penalty_excluded: out = out[out['is_excluded_from_penalties'] == False]; funnel['penalty_eligible'] = len(out)
    funnel['reverted_kept'] = int(out['reverted'].sum()); funnel['settled_kept'] = int((~out['reverted']).sum())
    return out.copy(), funnel

def reward_per_auction_solver(df):
    """Aggregate to (chain, auction, solver). Reward/penalty repeat across a multi-order solution's
    rows ('first'); order-level economics (gross pnl, volume) are summed."""
    g = df.groupby(['blockchain', 'auction_id', 'solver'], dropna=False)
    out = g.agg(
        solver_name=('solver_name', 'first'),
        reward_usd=('reward_usd', 'first'),
        penalty_usd=('penalty_usd', 'first'),
        penalty_uncapped_usd=('penalty_uncapped_usd', 'first'),
        reward_penalty_usd=('reward_penalty_usd', 'first'),
        penalty_cap_usd=('penalty_cap_usd', 'first'),
        gross_execution_pnl_usd=('gross_execution_pnl_usd', 'sum'),
        volume_usd=('volume_usd', 'sum'),
        size_usd=('size_usd', 'sum'),
        reverted=('reverted', 'any'),
        cap_binds=('cap_binds', 'any'),
        order_count=('order_uid', 'nunique'),
    ).reset_index()
    out['net_mechanism_pnl_usd'] = out['reward_penalty_usd'].fillna(0) + out['gross_execution_pnl_usd'].fillna(0)
    out['penalty_bps'] = (out['penalty_usd'] / out['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    out['reward_bps'] = (out['reward_usd'] / out['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    return out

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z**2/n; c = p + z**2/(2*n)
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))
    return (p, (c-h)/d, (c+h)/d)

FIXED_BINS = [0, 100, 1_000, 10_000, 100_000, np.inf]             
FIXED_LABELS = ['<$100', '$100-1k', '$1k-10k', '$10k-100k', '>$100k']

def add_fixed_buckets(df, col='size_usd'):
    df = df.copy()
    df['size_fixed'] = pd.cut(df[col], bins=FIXED_BINS, labels=FIXED_LABELS, include_lowest=True)
    return df

def add_chain_quantile_buckets(df, value='size_usd', q=5):
    """Per-chain quantile buckets (primary size view). duplicates='drop' + ordinal relabel so tie-heavy
    chains don't crash on fixed labels; ordered categorical so Q10 doesn't sort before Q2."""
    out = df.copy(); out['size_q'] = pd.Series(pd.NA, index=out.index, dtype='object')
    for ch, idx in out.groupby('blockchain').groups.items():
        s = out.loc[idx, value]; valid = s[s.notna() & np.isfinite(s) & (s > 0)]
        if len(valid) < q: continue
        try: binned = pd.qcut(valid, q=q, duplicates='drop')
        except ValueError: continue
        out.loc[valid.index, 'size_q'] = binned.cat.codes.map(lambda c: f'Q{c+1}')
    out['size_q'] = pd.Categorical(out['size_q'], categories=[f'Q{i}' for i in range(1, q+1)], ordered=True)
    return out

def ecdf(ax, series, label, color=None):
    x = np.sort(pd.Series(series).dropna().values)
    if len(x): ax.plot(x, np.arange(1, len(x)+1)/len(x), label=label, color=color)

def barpos(ax, labels, vals, **kw):
    """Bar with explicit numeric x + string labels (avoids matplotlib categorical-float errors)."""
    x = np.arange(len(labels)); ax.bar(x, np.asarray(vals, float), **kw)
    ax.set_xticks(x); ax.set_xticklabels([str(l) for l in labels])

print('helpers defined')

In [ ]:
print('loading:'); RAW = prepare(load_chain_csvs())
DF, FUNNEL = apply_filters(RAW)
print('\nfunnel:', {k: f'{v:,}' for k, v in FUNNEL.items()})
print('chains:', DF['blockchain'].unique().tolist())
MULTI = DF['blockchain'].nunique() > 1
if not MULTI: print('single chain')

Sanity panel before interpretation: per-chain coverage + solver HHI, null-on-revert check for settled-only columns, and p50/p99/max to catch native-unit / huge-size artifacts that dominate totals.

In [ ]:
def coverage_table(df):
    rows = []
    for ch, d in df.groupby('blockchain'):
        shares = d['solver'].value_counts(normalize=True)
        rows.append({'chain': ch, 'attempts': len(d), 'orders': d['order_uid'].nunique(),
            'solvers': d['solver'].nunique(), 'revert_rate': d['reverted'].mean(),
            'penalty_cap_usd': d['penalty_cap_usd'].median(), 'median_size_usd': d['size_usd'].median(),
            'solver_hhi': float((shares**2).sum()), 'top_solver_share': float(shares.iloc[0]),
            'token_pairs': (d['sell_token'].astype(str)+'/'+d['buy_token'].astype(str)).nunique()})
    return pd.DataFrame(rows).set_index('chain')

def missingness_check(df):
    cols = ['order_size_usd','markout_usd','markout_relative','seconds_to_settle','slippage_usd','execution_cost_native']
    rev, st = df[df['reverted']], df[~df['reverted']]
    return pd.DataFrame({'null_on_revert_%': [100*rev[c].isna().mean() for c in cols],
                         'null_on_settled_%': [100*st[c].isna().mean() for c in cols]}, index=cols).round(1)

def outlier_diag(df):
    cols = ['volume_tok','penalty_uncapped_tok','size_usd','penalty_uncapped_usd']
    return {ch: pd.DataFrame({c: [d[c].quantile(.5), d[c].quantile(.99), d[c].max()] for c in cols},
                             index=['p50','p99','max']) for ch, d in df.groupby('blockchain')}

print('=== coverage ==='); display(coverage_table(DF))
print('=== missingness (settled-only cols ~100% null on revert) ==='); display(missingness_check(DF))
print('=== outliers (p50/p99/max) ===')
for ch, t in outlier_diag(DF).items(): print(ch); display(t)

## 1. Revert rates by chain vs rewards & penalties
Per-chain revert rate (attempt grain, Wilson CI) with median reward/penalty (deduped to auction-solver grain).

In [ ]:
def revert_by_chain(df):
    rps = reward_per_auction_solver(df); rows = []
    for ch, d in df.groupby('blockchain'):
        p, lo, hi = wilson(int(d['reverted'].sum()), len(d))
        r = rps[rps['blockchain'] == ch]
        rows.append({'chain': ch, 'n_attempts': len(d), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi,
            'penalty_cap_usd': d['penalty_cap_usd'].median(), 'med_reward_usd': r['reward_usd'].median(),
            'med_penalty_usd': r['penalty_usd'].median(),
            'med_penalty_on_revert_usd': r.loc[r['reverted'], 'penalty_usd'].median()})
    return pd.DataFrame(rows).set_index('chain')

def plot_revert_by_chain(tbl):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    x = np.arange(len(tbl))

    lo = (tbl['revert_rate'] - tbl['ci_lo']).clip(lower=0).fillna(0)
    hi = (tbl['ci_hi'] - tbl['revert_rate']).clip(lower=0).fillna(0)

    ax[0].bar(
        x,
        tbl['revert_rate'] * 100,
        yerr=np.vstack([lo, hi]) * 100,
        capsize=4,
        color='#c0504d'
    )
    ax[0].set_xticks(x)
    ax[0].set_xticklabels([str(i) for i in tbl.index])
    ax[0].set_ylabel('revert rate (%)')
    ax[0].set_title('Revert rate by chain (95% Wilson CI)')

    w = 0.38
    ax[1].bar(
        x - w/2,
        tbl['med_reward_usd'],
        w,
        label='median reward',
        color='#4f81bd'
    )
    ax[1].bar(
        x + w/2,
        tbl['med_penalty_on_revert_usd'],
        w,
        label='median penalty on revert',
        color='#c0504d'
    )
    ax[1].set_xticks(x)
    ax[1].set_xticklabels([str(i) for i in tbl.index])
    ax[1].set_ylabel('USD, auction-solver grain')
    ax[1].set_title('Median reward vs median penalty on revert')
    ax[1].legend()

    plt.tight_layout()
    plt.show()

T2 = revert_by_chain(DF); display(T2); plot_revert_by_chain(T2)

## 2. Revert rate by order size
Primary: per-chain quantile buckets (scale-adaptive), with median/min/max USD per bucket so Q-labels stay economically interpretable. Secondary: fixed-$ buckets for cross-chain comparability. Visual: boundary-free rolling revert rate over log10(size).

In [ ]:
def revert_by_quantile(df):
    d = add_chain_quantile_buckets(df); rows = []
    for (ch, q), g in d.groupby(['blockchain','size_q'], observed=True):
        if len(g) == 0: continue
        p, lo, hi = wilson(int(g['reverted'].sum()), len(g))
        rows.append({'chain': ch, 'size_q': q, 'n': len(g), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi,
            'med_size_usd': g['size_usd'].median(), 'min_size_usd': g['size_usd'].min(), 'max_size_usd': g['size_usd'].max(),
            'med_penalty_bps': g.loc[g['reverted'], 'penalty_bps'].median(),
            'med_gross_pnl_usd': g['gross_execution_pnl_usd'].median()})
    return pd.DataFrame(rows)

def revert_by_fixed(df):
    d = add_fixed_buckets(df); rows = []
    for (ch, b), g in d.groupby(['blockchain','size_fixed'], observed=True):
        if len(g) == 0: continue
        p, lo, hi = wilson(int(g['reverted'].sum()), len(g))
        rows.append({'chain': ch, 'bucket': b, 'n': len(g), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi})
    return pd.DataFrame(rows)

def plot_revert_by_size(qtbl, df):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for ch, g in qtbl.groupby('chain'):
        g = g.set_index('size_q').reindex([f'Q{i}' for i in range(1, 6)])
        lo = (g['revert_rate']-g['ci_lo']).clip(lower=0).fillna(0); hi = (g['ci_hi']-g['revert_rate']).clip(lower=0).fillna(0)
        ax[0].errorbar(range(len(g)), g['revert_rate']*100, yerr=np.vstack([lo, hi])*100, marker='o', capsize=3, label=ch)
    ax[0].set_xticks(range(5)); ax[0].set_xticklabels([f'Q{i}' for i in range(1, 6)])
    ax[0].set_xlabel('per-chain size quintile'); ax[0].set_ylabel('revert rate (%)')
    ax[0].set_title('Revert rate by per-chain size quantile'); ax[0].legend()
    for ch, d in df.groupby('blockchain'):
        v = d[['size_usd','reverted']].dropna(); v = v[v['size_usd'] > 0]
        if len(v) < 50: continue
        lx = np.log10(v['size_usd']); bins = np.linspace(lx.min(), lx.max(), 25); idx = np.digitize(lx, bins)
        xs, ys = [], []
        for b in range(1, len(bins)):
            m = idx == b
            if m.sum() >= 20: xs.append(lx[m].median()); ys.append(v['reverted'][m].mean()*100)
        ax[1].plot(xs, ys, marker='.', label=ch)
    ax[1].set_xlabel('log10(size_usd)'); ax[1].set_ylabel('revert rate (%)')
    ax[1].set_title('Log-binned revert rate vs size (boundary-free)'); ax[1].legend()
    plt.tight_layout(); plt.show()
    
def plot_penalty_bps_by_fixed_size(df):
    fixed_order = ['<$100', '$100-1k', '$1k-10k', '$10k-100k', '>$100k']

    d = add_fixed_buckets(df.copy())
    d = d[d['reverted'] & d['penalty_bps'].notna()]

    tbl = (
        d.groupby(['size_fixed', 'blockchain'], observed=True)['penalty_bps']
        .median()
        .unstack('blockchain')
        .reindex(fixed_order)
    )

    display(tbl.round(2))

    fig, ax = plt.subplots(figsize=(9, 4.5))
    for ch in tbl.columns:
        ax.plot(tbl.index.astype(str), tbl[ch], marker='o', label=ch)

    ax.set_ylabel('median penalty on reverts (bps of size)')
    ax.set_xlabel('order-size bucket')
    ax.set_title('Fixed cap severity by order size')
    ax.tick_params(axis='x', rotation=25)
    ax.legend()
    plt.tight_layout()
    plt.show()

T4q = revert_by_quantile(DF); print('=== per-chain quantile (primary) ==='); display(T4q)
T4f = revert_by_fixed(DF); print('=== fixed-$ buckets (secondary) ===')
fixed_order = ['<$100', '$100-1k', '$1k-10k', '$10k-100k', '>$100k']
display(T4f.pivot_table(index='bucket', columns='chain', values='revert_rate', observed=True).reindex(fixed_order))
plot_revert_by_size(T4q, DF)
plot_penalty_bps_by_fixed_size(DF)

## 3. Cap vs economics (reward / both P&L concepts)
Per-chain median reward, gross P&L (execution-only) and net P&L (incl reward/penalty) vs penalty cap. Answers if a higher penalty cap correlate with lower rewards and lower solver PnL?

In [ ]:
rps = reward_per_auction_solver(DF)

econ = rps.groupby('blockchain').agg(
    med_reward_usd=('reward_usd', 'median'),
    med_penalty_usd_all=('penalty_usd', 'median'),
    med_net_pnl_usd=('net_mechanism_pnl_usd', 'median'),
    revert_rate_solver=('reverted', 'mean'),
    share_penalized=('penalty_usd', lambda s: (s > 0).mean()),
)

penalty_on_revert = (
    rps[rps['reverted']]
    .groupby('blockchain')['penalty_usd']
    .median()
    .rename('med_penalty_on_revert_usd')
)

caps = DF.groupby('blockchain').agg(
    penalty_cap_tok=('penalty_cap_tok', 'median'),
    penalty_cap_tok_unique=('penalty_cap_tok', 'nunique'),
    median_penalty_cap_usd=('penalty_cap_usd', 'median'),

    reward_cap_tok=('reward_cap_upper_tok', 'median'),
    reward_cap_tok_unique=('reward_cap_upper_tok', 'nunique'),
    median_reward_cap_usd=('reward_cap_upper_usd', 'median'),

    med_gross_pnl_usd=('gross_execution_pnl_usd', 'median'),
    revert_rate_attempt=('reverted', 'mean'),
    median_size_usd=('size_usd', 'median'),
    n_attempts=('order_uid', 'size'),
    n_orders=('order_uid', 'nunique'),
)

T5 = (
    caps
    .join(econ)
    .join(penalty_on_revert)
    .sort_values('median_penalty_cap_usd')
)
T5['reward_cap_minus_penalty_cap_usd'] = (
    T5['median_reward_cap_usd'] - T5['median_penalty_cap_usd']
)
T5['reward_cap_to_penalty_cap'] = (
    T5['median_reward_cap_usd'] / T5['median_penalty_cap_usd']
)
print('=== caps vs solver economics ===')
display(T5.round({
    'penalty_cap_tok': 4,
    'median_penalty_cap_usd': 4,
    'reward_cap_tok': 4,
    'median_reward_cap_usd': 4,
    'med_reward_usd': 4,
    'med_penalty_usd_all': 4,
    'med_penalty_on_revert_usd': 4,
    'med_gross_pnl_usd': 4,
    'med_net_pnl_usd': 4,
    'revert_rate_attempt': 4,
    'revert_rate_solver': 4,
    'share_penalized': 4,
    'median_size_usd': 2,
    'reward_cap_minus_penalty_cap_usd': 4,
    'reward_cap_to_penalty_cap': 4,
}))
if len(T5) == 1:
    print(
        f"Only one chain loaded ({T5.index[0]}): this section is a summary only. "
        "Load the second chain to compare cap direction."
    )

else:
    low_chain = T5.index[0]
    high_chain = T5.index[-1]

    direction = pd.DataFrame({
        'low_penalty_cap_chain': low_chain,
        'high_penalty_cap_chain': high_chain,
        'low_penalty_cap_value': T5.iloc[0],
        'high_penalty_cap_value': T5.iloc[-1],
        'high_minus_low': T5.iloc[-1] - T5.iloc[0],
    })[
        [
            'low_penalty_cap_chain',
            'high_penalty_cap_chain',
            'low_penalty_cap_value',
            'high_penalty_cap_value',
            'high_minus_low',
        ]
    ]

    direction = direction.loc[
        [
            'median_penalty_cap_usd',
            'median_reward_cap_usd',
            'med_reward_usd',
            'med_gross_pnl_usd',
            'med_net_pnl_usd',
            'revert_rate_attempt',
            'revert_rate_solver',
            'share_penalized',
            'med_penalty_on_revert_usd',
            'median_size_usd',
        ]
    ]

    if len(T5) > 2:
        print(
            f"{len(T5)} chains loaded: table sorted by median penalty cap USD; "
            "direction below compares the lowest-cap vs highest-cap chain."
        )
    print('=== direction from lower penalty cap chain to higher penalty cap chain ===')
    display(direction.round(4))

if len(T5) >= 2:
    plot_df = T5.sort_values('median_penalty_cap_usd')

    y_metrics = [
        ('med_reward_usd', 'Median reward'),
        ('med_gross_pnl_usd', 'Median gross PnL'),
        ('med_net_pnl_usd', 'Median net PnL'),
    ]

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))

    for a, (ycol, ylabel) in zip(ax, y_metrics):
        a.plot(
            plot_df['median_penalty_cap_usd'],
            plot_df[ycol],
            marker='o',
            ls='none',
        )

        for ch, r in plot_df.iterrows():
            label = (
                f"{ch}\n"
                f"reward cap=${r['median_reward_cap_usd']:.2f}"
            )
            a.annotate(
                label,
                (r['median_penalty_cap_usd'], r[ycol]),
                textcoords='offset points',
                xytext=(6, 6),
                fontsize=9,
            )

        a.axhline(0, linewidth=1, alpha=0.4)
        a.set_xlabel('Median penalty cap, USD')
        a.set_ylabel(f'{ylabel}, USD')
        a.set_title(f'{ylabel} vs penalty cap')

    plt.tight_layout()
    plt.show()

## 4. Cap vs markout (surplus delivered to users)
Markout is measured in bps on settled executions; test whether higher-penalty-cap chains show worse median markout, with ECDFs comparing the full clipped distribution.

In [ ]:
def markout_vs_cap(df):
    d = add_fixed_buckets(df[df['settled'] == True])
    by_chain = d.groupby('blockchain').agg(penalty_cap_usd=('penalty_cap_usd','median'), reward_cap_usd=('reward_cap_upper_usd', 'median'),
        med_markout_bps=('markout_bps','median'), n=('order_uid','size'), n_markout=('markout_bps', 'count'),
        median_size_usd=('size_usd', 'median')).sort_values('penalty_cap_usd')
    by_size = d.groupby(['blockchain','size_fixed'], observed=True)['markout_bps'].median().unstack(0)
    return by_chain, by_size, d

def plot_markout(by_chain, d):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    if len(by_chain) >= 2:
        p = by_chain.sort_values('penalty_cap_usd')

        ax[0].plot(
            p['penalty_cap_usd'],
            p['med_markout_bps'],
            marker='o',
            ls='none',
            color='#9bbb59'
        )

        for ch, r in p.iterrows():
            ax[0].annotate(
                f"{ch}\nreward cap=${r['reward_cap_usd']:.2f}",
                (r['penalty_cap_usd'], r['med_markout_bps']),
                textcoords='offset points',
                xytext=(6, 6),
                fontsize=9,
            )

        ax[0].set_xlabel('median penalty cap (USD)')
        ax[0].set_ylabel('median markout (bps)')
        ax[0].set_title('Median markout vs penalty cap')
    else:
        barpos(ax[0], list(by_chain.index), by_chain['med_markout_bps'], color='#9bbb59')
        ax[0].set_xlabel('chain')
        ax[0].set_ylabel('median markout (bps)')
        ax[0].set_title('Surplus to users by chain')

    ax[0].axhline(0, color='k', lw=.8)

    for ch, g in d.groupby('blockchain'):
        ecdf(ax[1], g['markout_bps'].clip(-200, 200), ch)

    ax[1].set_xlabel('markout (bps, clipped ±200)')
    ax[1].set_ylabel('ECDF')
    ax[1].set_title('Markout distribution')
    ax[1].legend()

    plt.tight_layout()
    plt.show()

T6c, T6s, T6d = markout_vs_cap(DF); display(T6c); display(T6s); plot_markout(T6c, T6d)

## 5. Three outcomes vs cap
Compare solver gross execution P&L, settled-user markout, and time-to-happy-moo against penalty cap across chains. Hypothesis: higher caps are associated with worse markout and longer time-to-happy-moo.

In [ ]:
def three_outcomes(df):
    st = df[df['settled'] == True]
    base = df.groupby('blockchain').agg(penalty_cap_usd=('penalty_cap_usd','median'), reward_cap_usd=('reward_cap_upper_usd', 'median'),
        med_gross_pnl_usd=('gross_execution_pnl_usd','median'))
    return base.join([st.groupby('blockchain')['markout_bps'].median().rename('med_markout_bps'),
        st.groupby('blockchain')['seconds_to_settle'].median().rename('med_tthm_s'), st.groupby('blockchain')['markout_bps'].count().rename('n_markout'),
        st.groupby('blockchain')['seconds_to_settle'].count().rename('n_tthm'),]).sort_values('penalty_cap_usd')

def plot_three(tbl):
    metrics = [('med_gross_pnl_usd','solver gross P&L (USD)'), ('med_markout_bps','markout (bps, user)'),
               ('med_tthm_s','time-to-happy-moo (s)'), ('med_tthm_blocks', 'time-to-happy-moo (blocks)')]
    fig, ax = plt.subplots(1, 4, figsize=(20, 4.5))
    for (m, lab), a in zip(metrics, ax):
        if len(tbl) >= 2:
            p = tbl.sort_values('penalty_cap_usd')
            a.plot(p['penalty_cap_usd'], p[m], marker='o', ls='none')
            for ch, r in p.iterrows():
                a.annotate(
                    f"{ch}\nreward cap=${r['reward_cap_usd']:.2f}",
                    (r['penalty_cap_usd'], r[m]),
                    textcoords='offset points',
                    xytext=(6, 6),
                    fontsize=9,
                )
            a.set_xlabel('penalty cap (USD)')
        else: 
            barpos(a, list(tbl.index), tbl[m])
            a.set_xlabel('chain')
        a.axhline(0, color='k', lw=.8, alpha=.4); a.set_ylabel(lab); a.set_title(lab)
    plt.tight_layout(); plt.show()

BLOCK_SECONDS = {
    'polygon': 1.75,
    'bnb': 0.5,
    'ethereum': 12.0,
    'gnosis': 5.0,
    'base': 2.0,
    'avalanche_c': 1.0,
    'arbitrum': 0.25
}
T7 = three_outcomes(DF); display(T7.round({
    'penalty_cap_usd': 4,
    'reward_cap_usd': 4,
    'med_gross_pnl_usd': 4,
    'med_markout_bps': 2,
    'med_tthm_s': 2,
})) 
T7['med_tthm_blocks'] = (
    T7['med_tthm_s'] / T7.index.to_series().map(BLOCK_SECONDS)
)
plot_three(T7)

## 6. Distribution of capped vs uncapped penalties, per chain.
On reverts: how often the cap binds and how much penalty it forgives. 

In [ ]:
def cap_binding(df):
    rev = reward_per_auction_solver(df); rev = rev[rev['reverted']]; rows = []
    for ch, d in rev.groupby('blockchain'):
        unc = d['penalty_uncapped_usd']; cap = d['penalty_usd']; w = unc.quantile(0.999)
        rows.append({'chain': ch, 'reverts': len(d), 'cap_binds_rate': d['cap_binds'].mean(),
            'med_penalty_on_revert_usd': cap.median(), 'capped_sum_usd': cap.sum(), 'uncapped_sum_usd': unc.sum(),
            'forgiven_sum_usd': (unc-cap).sum(), 'forgiven_sum_winsor_usd': (np.minimum(unc, w)-cap).sum()})
    return pd.DataFrame(rows).set_index('chain')

def cap_percentile(df):
    rps = reward_per_auction_solver(df)
    rev = rps[rps['reverted'] & (rps['penalty_uncapped_usd'] > 0) & (rps['penalty_cap_usd'] > 0)]
    rows = []
    for ch, d in rev.groupby('blockchain'):
        unc = d['penalty_uncapped_usd'].values
        cap = d['penalty_cap_usd'].median()
        rows.append(dict(chain=ch, cap_usd=round(cap, 2),
                         cap_percentile=round((unc <= cap).mean() * 100, 1),
                         cap_binds_rate=round((unc > cap).mean() * 100, 1),
                         unc_p50=round(np.median(unc), 2),
                         unc_p90=round(np.quantile(unc, .9), 2),
                         unc_p99=round(np.quantile(unc, .99), 2)))
    return pd.DataFrame(rows).set_index('chain')

def plot_cap_binding(df, tbl):
    rev = reward_per_auction_solver(df); rev = rev[rev['reverted'] & (rev['penalty_uncapped_usd'] > 0)]
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    chains = list(rev['blockchain'].unique())
    cmap = {c: col for c, col in zip(chains, plt.cm.tab10(np.linspace(0, 1, max(len(chains), 1))))}
    for ch, d in rev.groupby('blockchain'):
        ecdf(ax[0], np.log10(d['penalty_uncapped_usd'].clip(lower=1e-6)), f'{ch} uncapped', color=cmap[ch])
        ax[0].axvline(np.log10(d['penalty_cap_usd'].median()), ls='--', alpha=.6, color=cmap[ch])
    ax[0].set_xlabel('log10 uncapped penalty (USD)'); ax[0].set_ylabel('ECDF')
    ax[0].set_title('Uncapped penalty vs cap (dashed)'); ax[0].legend(fontsize=8)
    barpos(ax[1], list(tbl.index), tbl['cap_binds_rate']*100, color='#c0504d')
    ax[1].set_ylabel('% reverts where cap binds'); ax[1].set_title('Cap-binding rate')
    plt.tight_layout(); plt.show()
def plot_uncapped_over_cap(df):
    rps = reward_per_auction_solver(df).copy()
    rev = rps[
        rps['reverted']
        & (rps['penalty_uncapped_usd'] > 0)
        & (rps['penalty_cap_usd'] > 0)
    ].copy()
    rev['uncapped_over_cap'] = (
        rev['penalty_uncapped_usd'] / rev['penalty_cap_usd']
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    for ch, d in rev.groupby('blockchain'):
        ecdf(ax, d['uncapped_over_cap'].clip(lower=1e-6), ch)
    ax.axvline(1, color='k', ls='--', lw=1, alpha=.7)
    ax.set_xscale('log')
    ax.set_xlabel('uncapped penalty / cap')
    ax.set_ylabel('ECDF')
    ax.set_title('Uncapped penalty relative to cap')
    ax.legend()

    plt.tight_layout()
    plt.show()
T8 = cap_binding(DF); display(T8); plot_cap_binding(DF, T8); plot_uncapped_over_cap(DF)
T8b = cap_percentile(DF)
print("Cap percentile placement:")
display(T8b)

## 7. Variable-rate penalty counterfactual
Recompute penalties had the cap been a variable rate (% of size): `min(uncapped, r x volume)`: a variable cap, bidding held constant. Report collected penalty across a grid of rates, the revenue-neutral rate per chain (exact, capped bisection), and who pays more/less by size band and solver.

In [ ]:
def variable_rate(df, grid=(1, 2, 5, 10, 20, 50)):
    rev = reward_per_auction_solver(df)
    rev = rev[rev['reverted'] & rev['size_usd'].notna() & (rev['size_usd'] > 0)].copy()
    rev = rev[rev.groupby('blockchain')['size_usd'].transform(lambda s: s < s.quantile(0.999))]  
    def collected(unc, size, r):                       
        return np.minimum(unc, r / 1e4 * size).sum()
    def neutral_bps(unc, size, target):                # exact rev-neutral (sum is monotone in r)
        lo, hi = 0.0, 1000.0
        for _ in range(60):
            mid = (lo + hi) / 2
            lo, hi = (mid, hi) if collected(unc, size, mid) < target else (lo, mid)
        return (lo + hi) / 2
    sweep_rows, neutral, parts = [], {}, []
    for ch, d in rev.groupby('blockchain'):
        unc, size, current = d.penalty_uncapped_usd.values, d.size_usd.values, d.penalty_usd.sum()
        for r in grid:
            sweep_rows.append(dict(chain=ch, rate_bps=r, total_usd=collected(unc, size, r),
                                   ratio=collected(unc, size, r) / current if current else np.nan))
        neutral[ch] = neutral_bps(unc, size, current)
        parts.append(d.assign(pen_fixed=d.penalty_usd,
                              pen_var=np.minimum(d.penalty_uncapped_usd, neutral[ch] / 1e4 * d.size_usd)))
    gw = pd.concat(parts)
    bands  = [0, 100, 1_000, 10_000, 100_000, np.inf]
    labels = ["<$100", "$100-1k", "$1k-10k", "$10k-100k", ">$100k"]
    gw["band"] = pd.cut(gw.size_usd, bands, labels=labels)
    return pd.DataFrame(sweep_rows), neutral, gw, labels

def share_shift(gw, key, chain=None):
    d = gw if chain is None else gw[gw.blockchain == chain]
    tf, tv = d.pen_fixed.sum(), d.pen_var.sum()
    s = d.groupby(key, observed=True).agg(fixed=("pen_fixed", "sum"), variable=("pen_var", "sum"))
    s["fixed_share"], s["var_share"] = s.fixed / tf, s.variable / tv
    s["delta_pp"] = (s.var_share - s.fixed_share) * 100
    return s

def plot_variable(sweep, neutral, gw, labels):
    chains = list(neutral)
    colors = {ch: c for ch, c in zip(chains, plt.cm.tab10(np.linspace(0, 1, max(len(chains), 1))))}
    fig, ax = plt.subplots(2, 2, figsize=(14, 9))
    for ch in chains:
        d = sweep[sweep.chain == ch]
        ax[0,0].plot(d.rate_bps, d.ratio, marker='o', color=colors[ch],
                     label=f"{ch} (neutral ≈ {neutral[ch]:.1f} bps)")
        ax[0,0].axvline(neutral[ch], color=colors[ch], ls=':', lw=1)
    ax[0,0].axhline(1.0, color='gray', ls='--', lw=.8)
    ax[0,0].set(xlabel="rate (bps of size)", ylabel="collected / current",
                title="Variable rate vs current flat cap\n(1.0 = revenue neutral; dotted = neutral rate)")
    ax[0,0].legend()
    for ch in chains:
        d = gw[gw.blockchain == ch].copy()
        if len(d) < 200: continue
        d["dec"] = pd.qcut(d.size_usd, 10, labels=False, duplicates="drop")
        eff = d.groupby("dec").apply(lambda x: x.pen_fixed.sum() / x.size_usd.sum() * 1e4,
                                     include_groups=False)
        msz = d.groupby("dec").size_usd.median()
        ax[0,1].plot(msz.values, eff.values, 'o-', color=colors[ch], label=ch)
    for ch in chains:
        ax[0,1].axhline(neutral[ch], color=colors[ch], ls=':', lw=1)
    ax[0,1].set(xscale="log", yscale="log", xlabel="median order size in decile (USD)",
                ylabel="effective penalty (bps of size)",
                title="Current fixed cap is REGRESSIVE\n(flat rate = horizontal; dotted = neutral rate)")
    ax[0,1].legend()
    bd = pd.DataFrame({ch: share_shift(gw, "band", ch)["delta_pp"].reindex(labels)
                       for ch in chains})
    bd.plot.bar(ax=ax[1,0], color=[colors[ch] for ch in chains], width=.8)
    ax[1,0].axhline(0, color='k', lw=.8)
    ax[1,0].set(ylabel="budget-share shift (pp)", xlabel="order size band",
                title="Who pays more under the rate (variable − fixed)\n+ve = pays more")
    ax[1,0].tick_params(axis='x', rotation=30); ax[1,0].legend(title=None)
    gs = share_shift(gw, "solver_name").assign(n=gw.groupby("solver_name").size())
    gs = gs[gs.n >= 20].sort_values("delta_pp")
    pick = pd.concat([gs.head(5), gs.tail(5)])
    bar_c = ['#2f7d62' if v < 0 else '#b1442f' for v in pick.delta_pp]
    ax[1,1].barh(range(len(pick)), pick.delta_pp, color=bar_c)
    ax[1,1].set_yticks(range(len(pick))); ax[1,1].set_yticklabels(pick.index, fontsize=8)
    ax[1,1].axvline(0, color='k', lw=.8)
    ax[1,1].set(xlabel="budget-share shift (pp)",
                title="Solver winners (green) & losers (red)\nunder revenue-neutral rate")
    plt.tight_layout(); plt.show()
sweep, neutral, gw, labels = variable_rate(DF)
print("revenue-neutral rate (bps of size, capped):", {k: round(v, 2) for k, v in neutral.items()})
display(sweep.round(2))
plot_variable(sweep, neutral, gw, labels)

## 8. Sit-out (non-economic penalty) simulation
Suspend a reverting solver for N auctions (exponential backoff, `block_deadline` order): tally forgone reward, removed attempts, coverage-at-risk. The triggering revert is still penalised, future in-window penalties are a diagnostic only (not credited). Does not change overbidding incentives.

In [ ]:
def sit_out(df, base=1, factor=2, max_power=5, reset_clean=10):
    cap = factor ** max_power
    rps = reward_per_auction_solver(df)
    bd = (df.groupby(['blockchain', 'auction_id', 'solver'])['block_deadline']
            .first().reset_index())
    base_df = rps.merge(bd, on=['blockchain', 'auction_id', 'solver'], how='left')
    settled = base_df[~base_df['reverted']]
    osettle = ({ch: {o: list(zip(gg['block_deadline'], gg['solver']))
                     for o, gg in g.groupby('order_uid')}
                for ch, g in settled.groupby('blockchain')}
               if 'order_uid' in base_df else {})

    res = []
    for (ch, solver), d in base_df.sort_values('block_deadline').groupby(['blockchain', 'solver']):
        d = d.reset_index(drop=True)
        rev = d['reverted'].to_numpy()
        rew = d['reward_usd'].fillna(0).to_numpy(float)
        pen = d['penalty_usd'].fillna(0).to_numpy(float)
        oc  = d['order_count'].fillna(1).to_numpy() if 'order_count' in d else np.ones(len(d))
        oid = d['order_uid'].to_numpy() if 'order_uid' in d else np.array([None] * len(d))
        bdl = d['block_deadline'].to_numpy(float)

        ban = offense = clean = 0
        trig_pen = forg = future_pen_diag = 0.0
        removed = removed_settled = removed_rev = covg = 0

        for i in range(len(d)):
            if ban > 0:                                  # suspended: drop this attempt
                ban -= 1; removed += int(oc[i])
                if not rev[i]:
                    removed_settled += 1; forg += rew[i]
                    covered = any(o_bd > bdl[i] and o_sv != solver
                                  for o_bd, o_sv in osettle.get(ch, {}).get(oid[i], []))
                    if not covered: covg += int(oc[i])
                else:
                    removed_rev += 1; future_pen_diag += pen[i]   
                continue
            if rev[i]:                                   # active revert: trigger / escalate
                offense += 1; clean = 0
                ban = int(min(cap, base * factor ** (offense - 1)))
                trig_pen += pen[i]                       # triggering penalty still incurred
            else:                                        # active clean attempt: maybe reset
                clean += 1
                if clean >= reset_clean: offense = clean = 0

        res.append({
            'chain': ch,
            'solver': d['solver_name'].iloc[0] if d['solver_name'].notna().any() else solver,
            'reverts': int(rev.sum()),
            'triggering_penalty_usd': trig_pen,
            'forgone_reward_usd': forg,
            'removed_orders': removed,
            'removed_settled_wins': removed_settled,
            'removed_reverted_wins': removed_rev,
            'coverage_at_risk_orders_ub': covg,
            'future_penalty_diag_usd': future_pen_diag,  
        })

    out = pd.DataFrame(res)
    out['sitout_cost_usd'] = out['forgone_reward_usd']  
    return out.sort_values('sitout_cost_usd', ascending=False)

def plot_sit_out(tbl, top=15):
    t = tbl.head(top)
    labels = [f"{c} / {s}" for c, s in zip(t['chain'], t['solver'])][::-1]
    y = np.arange(len(labels)); w = 0.4
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.barh(y + w/2, t['forgone_reward_usd'][::-1], w, label='forgone reward', color='#8064a2')
    ax.barh(y - w/2, t['coverage_at_risk_orders_ub'][::-1], w, label='coverage-at-risk (orders, ub)', color='#4f81bd')
    ax.set_yticks(y); ax.set_yticklabels(labels)
    ax.axvline(0, color='k', lw=.8)
    ax.set_xlabel('per-solver sit-out cost')
    ax.set_title(f'Sit-out opportunity cost by chain/solver (top {top})')
    ax.legend(); plt.tight_layout(); plt.show()
    
SCHEDULES = {
    "1 auction (flat)":   dict(base=1, factor=1, max_power=0),   
    "1,2,4,... cap 32":   dict(base=1, factor=2, max_power=5),   
    "1,3,9,... cap 81":   dict(base=1, factor=3, max_power=4),   
    "1,2,4,... cap 8":    dict(base=1, factor=2, max_power=3),   
}

rows = []
detail = {}
for name, kw in SCHEDULES.items():
    t = sit_out(DF, **kw); detail[name] = t
    g = t.groupby('chain').agg(
        solvers=('solver', 'nunique'),
        reverts=('reverts', 'sum'),
        triggering_penalty_usd=('triggering_penalty_usd', 'sum'),
        forgone_reward_usd=('forgone_reward_usd', 'sum'),
        removed_orders=('removed_orders', 'sum'),
        coverage_at_risk_orders_ub=('coverage_at_risk_orders_ub', 'sum'),
        future_penalty_diag_usd=('future_penalty_diag_usd', 'sum'),
    ).reset_index()
    g.insert(0, 'schedule', name)
    rows.append(g)

sit_sweep = pd.concat(rows, ignore_index=True)
display(sit_sweep.round(2))
plot_sit_out(detail["1 auction (flat)"])

## 9. Within-solver paired comparison (cross-chain)
Same solver on different-cap chains, controls for solver skill/infra/flow, cross-chain confounder. Needs ≥2 chains.

In [ ]:
def within_solver(df, min_n=None):
    g = (
        df.groupby(['solver_name', 'blockchain'], dropna=False)
        .agg(
            n=('order_uid', 'size'),
            revert_rate=('reverted', 'mean'),
            med_gross_pnl_usd=('gross_execution_pnl_usd', 'median'),
            med_net_pnl_usd=('net_mechanism_pnl_usd', 'median'),
            med_markout_bps=('markout_bps', 'median'),
            penalty_cap_usd=('penalty_cap_usd', 'median'),
        )
        .reset_index()
    )
    if min_n is not None:
        g = g[g['n'] >= min_n].copy()
    multi = g.groupby('solver_name')['blockchain'].transform('nunique') > 1
    return (
        g[multi]
        .sort_values(['solver_name', 'penalty_cap_usd'])
        .reset_index(drop=True)
    )
def plot_within_solver(tbl):
    if tbl.empty or tbl['blockchain'].nunique() < 2:
        print('Need solvers active on ≥2 chains.')
        return
    fig, ax = plt.subplots(1, 3, figsize=(17, 5))
    solvers = list(tbl['solver_name'].unique())
    cmap = {sv: col for sv, col in zip(solvers, plt.cm.tab20(np.linspace(0, 1, max(len(solvers), 1))))}
    for sv, g in tbl.groupby('solver_name'):
        g = g.sort_values('penalty_cap_usd')
        ax[0].plot(
            g['penalty_cap_usd'],
            g['revert_rate'] * 100,
            marker='o',
            color=cmap[sv],
            label=sv,
        )
        ax[1].plot(
            g['penalty_cap_usd'],
            g['med_gross_pnl_usd'],
            marker='o',
            color=cmap[sv],
            label=sv,
        )
        ax[2].plot(
            g['penalty_cap_usd'],
            g['med_markout_bps'],
            marker='o',
            color=cmap[sv],
            label=sv,
        )
    ax[0].set_ylabel('revert rate (%)')
    ax[1].set_ylabel('median gross P&L (USD)')
    ax[2].set_ylabel('median markout (bps)')
    ax[0].set_title('Within-solver revert rate vs cap')
    ax[1].set_title('Within-solver P&L vs cap')
    ax[2].set_title('Within-solver markout vs cap')
    for a in ax:
        a.set_xlabel('penalty cap (USD)')

    handles, labels = ax[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.0, 0.5),
               fontsize=7, ncol=1 + len(solvers) // 20, title='solver')
    plt.tight_layout(rect=(0, 0, 0.98, 1))
    plt.show()
T11 = within_solver(DF)
display(T11.head(20).round({
    'revert_rate': 4,
    'med_gross_pnl_usd': 4,
    'med_net_pnl_usd': 4,
    'med_markout_bps': 2,
    'penalty_cap_usd': 4,
}))

plot_within_solver(T11)

## 10. Headline table
One compact per-chain summary for export.

In [ ]:
def headline(df):
    cov = coverage_table(df); rv = revert_by_chain(df); cb = cap_binding(df)
    st = df[df['settled'] == True].groupby('blockchain')
    return pd.DataFrame({'attempts': cov['attempts'], 'orders': cov['orders'], 'solvers': cov['solvers'],
        'revert_rate': rv['revert_rate'], 'ci_lo': rv['ci_lo'], 'ci_hi': rv['ci_hi'],
        'penalty_cap_usd': cov['penalty_cap_usd'], 'cap_binds_rate': cb['cap_binds_rate'],
        'median_size_usd': cov['median_size_usd'], 'med_markout_bps': st['markout_bps'].median(),
        'med_tthm_s': st['seconds_to_settle'].median(), 'solver_hhi': cov['solver_hhi']}).round(3)

display(headline(DF))
print('\nfunnel:', {k: f'{v:,}' for k, v in FUNNEL.items()})
if not MULTI: print('Single chain')